# **Annotate a molecular dynamics description with LLM 📝**

### 🎯 Objectives:

- Annotate a description of a datasats or a Materials and Methods section of a molecular dynamics study.
- Identify key experimental procedures, parameters, and computational tools like MOLECULE (`MOL`), FORCEFIELD (`FFM`), SIMULATION_TIME (`STIME`), TEMPERATURE (`TEMP`), SOFTWARE NAME (`SOFTNAME`) and SOFTWARE VERSION (`SOFTVERS`).
- Control the output format of the annotations with `instructor` framework to reduce hallucinations.
- Visualize the annotation results for clarity and verification.


## Load libraries


In [ ]:
import os
from importlib.resources import files

from dotenv import load_dotenv

from mdner_llm.chore.extract_entities import annotate_with_instructor
from mdner_llm.models.entities import ListOfEntities
from mdner_llm.models.entities_with_positions import ListOfEntitiesPositions
from mdner_llm.utils.visualize_annotations import visualize_llm_annotation


/home/tess01hp/Desktop/mdner_llm/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
%load_ext watermark
%watermark

Last updated: 2026-02-28T22:14:41.295071+01:00

Python implementation: CPython
Python version       : 3.13.0
IPython version      : 8.13.2

Compiler    : Clang 18.1.8 
OS          : Linux
Release     : 6.8.0-51-generic
Machine     : x86_64
Processor   : x86_64
CPU cores   : 4
Architecture: 64bit



## Step 1: Define the text to annotate


In [3]:
TEXT_TO_ANNOTATE = """Molecular Dynamics Simulations

MD simulations were performed using GROMACS 4.0.7 software [17] with the OPLS-AA
force-field [18].L33 and P33 forms of β3 were soaked in a rhombic dodecahedral
simulation box with 60,622 TIP3P water molecules and 28 Na+ ions.
The distance between any atom of the protein and the box edges was set to at least 10 Å.
The total energy of the system was minimized twice (before and after the addition of the
ions) with a steepest descent algorithm. MD simulations were run under the NPT
thermodynamic ensemble and periodic boundary conditions were applied in all directions.
We used the weak coupling algorithm of Berendsen [19] to maintain the system at a
constant physiological temperature of 310 K using a coupling constant of 0.1 ps (protein
and water ions separately). Pressure was held constant using the Berendsen algorithm
[19] at 1 atm with a coupling constant of 1 ps. Water molecules were kept rigid using
the SETTLE algorithm [20]. All other bond lengths were constrained with the LINCS
algorithm [21], allowing a 2 fs time step. We used a short-range coulombic and van der
Waals cut-off of 10 Å and calculated the long-range electrostatic interactions using
the smooth particle mesh Ewald (PME) algorithm [22], [23] with a grid spacing of 1.2 Å
and an interpolation order of 4. The neighbor list was updated every 10 steps. After a
1 ns equilibration (with position restraints on the protein), each system was simulated
for 50 ns. For both systems, five 50 ns simulations were performed (using different
initial velocities) and one was extended until 100 ns for a total simulation time of
300 ns. Molecular conformations were saved every 100 ps for further analysis.
"""

## Step 2: Select the model


But before, we retrieve the API key for the openrouter API from the environment variables. Make sure to set the `OPENROUTER_API_KEY` environment variable before running this cell.


In [4]:
# Set the openrouter API key
load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")

In [5]:
# Choices:
# Docs: https://openrouter.ai/models
MODELS_OPENROUTER = [
    "openai/gpt-5.2",
    "openai/gpt-4o",
    "openai/gpt-oss-120b",
    "meta-llama/llama-4-maverick",
    "google/gemini-3-pro-preview",
    "mistralai/mistral-large-2512",
    "qwen/qwen3-vl-30b-a3b-thinking",
    "deepseek/deepseek-v3.2",
]

# We will use "openai/gpt-5.2" for annotation
SELECTED_MODEL = "openai/gpt-5.2"

## Step 3: Load the prompt template


In [6]:
prompt = (
    files("mdner_llm.prompt_templates")
    .joinpath("json_few_shot.txt")
    .read_text(encoding="utf-8")
)
print(f"{prompt[:400]}...")

# Named-Entity Recognition task

## Role definition

You are a highly specialized **Named Entity Recognition (NER) engine** dedicated to extracting entities from Molecular Dynamics (MD) simulation dataset descriptions or research articles.
Your SOLE function is to analyze the input text and output a valid JSON object containing only the extracted entities and their labels.

## Specifications

### ...


## Step 4: Annotate the text


In [7]:
response, elapsed_time = annotate_with_instructor(
    TEXT_TO_ANNOTATE,
    SELECTED_MODEL,
    API_KEY,
    prompt,
    # or ListOfEntitiesPositions
    # if you want to include positions
    # of the entities in the text
    response_model=ListOfEntities,
)
print(f"Annotation by {SELECTED_MODEL} completed in {elapsed_time:.2f} seconds:")
response

Annotation by openai/gpt-5.2 completed in 15.40 seconds:


ListOfEntities(entities=[SoftwareName(label='SOFTNAME', text='GROMACS'), SoftwareVersion(label='SOFTVERS', text='4.0.7'), ForceField(label='FFM', text='OPLS-AA'), Molecule(label='MOL', text='L33'), Molecule(label='MOL', text='P33'), Molecule(label='MOL', text='β3'), ForceField(label='FFM', text='TIP3P'), Molecule(label='MOL', text='water'), Molecule(label='MOL', text='Na'), Temperature(label='TEMP', text='310 K'), SimulationTime(label='STIME', text='1 ns'), SimulationTime(label='STIME', text='50 ns'), SimulationTime(label='STIME', text='50 ns'), SimulationTime(label='STIME', text='100 ns'), SimulationTime(label='STIME', text='300 ns')])

## Step 4: Visualize the annotation results


In [8]:
visualize_llm_annotation(response, TEXT_TO_ANNOTATE)

🧐 VISUALIZATION OF ENTITIES 
